In [ ]:
# Load the vars
import pickle
from utils.filemanager import get_latest_temp_subfolder

data_file = "_experiments/Sabater1d/alpha_vari/mean_only_data.pkl"
data_file = r'latest'
if data_file == "latest":
    latest_temp_folder = get_latest_temp_subfolder()
    data_file = latest_temp_folder + "/ro_data.pkl"
    print(f"loading: {data_file}")

loaded_objects = []
with open(f'{data_file}', 'rb') as f:
    try:
        while True:
            loaded_objects.append(pickle.load(f))
    except EOFError:
        pass

ro_data, model, gp_model, args = loaded_objects
ref_x, ref_q = model.load_opt(measure = args.measure, tau = args.tau, alpha = args.alpha)
ref_plot = model.load_domain_plot(measure = args.measure, tau = args.tau, alpha = args.alpha)
ref_domain, ref_q_vec = ref_plot["x"], ref_plot["J"]
ref_x_vec = ref_domain.nodes

### Design space quantile prediction

In [ ]:
import matplotlib.pyplot as plt
import numpy as np 
ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

n_ER = ro_data.EI_infill["n"]
cand = np.array(ro_data.EI_infill["cand_x"])
n_col = 3
n_row = int(np.ceil(n_ER / n_col))
idx = 0
EIvals = []

for i in range(n_row):
    fig, ax = plt.subplots(2, n_col, figsize = [12,3])

    for j in range(n_col):
        if idx+1 > n_ER:
            break
        plt_data = ro_data.EI_infill["plot_data"][idx]
        EIvals.append(plt_data["EI_star"])
        upper_bound = plt_data["J_mean_vec"][:,0] + plt_data["J_std_vec"][:,0]
        lower_bound = plt_data["J_mean_vec"][:,0] - plt_data["J_std_vec"][:,0]
        h3 = ax[0,j].fill_between(plt_data["x_domain"].nodes[:,0], upper_bound, lower_bound, color=ccycle[0], alpha=0.3, zorder = 50)
        h2 = ax[0,j].plot(plt_data["x_domain"].nodes, plt_data["J_mean_vec"][:,0], color=ccycle[0], zorder = 60)
        h1 = ax[0,j].plot(ref_x_vec, ref_q_vec, label = f"q_{args.tau}", color = "dimgray", zorder = 1)
        ymin, ymax = ax[0,j].get_ylim()
        h4 = ax[0,j].scatter(cand[:idx,0], (np.ones(len(cand[:idx,0]))*ymin)*0.9, marker = "+", color = "black", linewidth=1, s = 40, label = f"x infill", zorder = 100, clip_on=False)

        ax[0,j].set_title(f"EI Iteration {idx}")
        h5 = ax[1,j].plot(plt_data["x_domain"].nodes, plt_data["JEI"].numpy(), color = ccycle[2], linestyle='', marker='o', markersize=2, alpha = 0.25)
        h7 = ax[1,j].plot(plt_data["x_domain"].nodes, plt_data["JEI_sampling"].numpy(), color = ccycle[3], linestyle='', marker='o', markersize=2, alpha = 0.25)
        h6 = ax[1,j].scatter(plt_data["x_c"], plt_data["EI_star"], marker = "*", label = "max QEI DiffEvo", color = ccycle[2], zorder = 100, clip_on=False)

        idx += 1
    fig.tight_layout()

fig, ax = plt.subplots()
ax.plot(EIvals)


### TR infill iteration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np 
ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

n_TR = ro_data.TR_infill["n"]
cand = np.array(ro_data.TR_infill["cand_x"])
n_col = 3
n_row = int(np.ceil(n_TR / n_col))
idx = 0

for i in range(n_row):
    fig, ax = plt.subplots(1, n_col, figsize = [12,3])

    for j in range(n_col):
        if idx+1 > n_TR:
            break
        plt_data = ro_data.TR_infill["plot_data"][idx]

        lb = plt_data["trust_reg"][0,0]
        rb = plt_data["trust_reg"][1,0]

        ax[j].axvspan(lb, rb, color='red', alpha=0.3, label='Trust region')
        ax[j].scatter(plt_data["x_star"], plt_data["J_star"], marker = "*")
        ax[j].scatter(plt_data["x_c"], plt_data["J_c"], marker = "+")
        h2 = ax[j].plot(plt_data["x_domain"].nodes[:,0], plt_data["J"], color=ccycle[0], zorder = 60)
        h1 = ax[j].plot(ref_x_vec, ref_q_vec, label = f"q_{args.tau}", color = "dimgray", zorder = 1)
        ymin, ymax = ax[j].get_ylim()
        h4 = ax[j].scatter(cand[:idx,0], np.ones(len(cand[:idx,0]))*ymin-1.0, marker = "+", color = "black", linewidth=1, s = 40, label = f"x infill", zorder = 100, clip_on=False)

        idx += 1
    fig.tight_layout()

In [ ]:
import torch
torch.vstack(ro_data.TR_infill["best_x"])

### Candidate evolution

In [ ]:
from utils.plotting import plot_cand_evolution

plot_cand_evolution(ro_data, model, args)

### Best point evolution

In [ ]:
from utils.plotting import plot_best_evolution

plot_best_evolution(ro_data, model, args)

### Convergence

In [ ]:
from utils.plotting import plot_convergence

plot_convergence(ro_data=ro_data, model = model, args = args)